In [1]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [15]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_bm25_with_onboarding,
    full_dense_with_onboarding,
    full_hybrid_with_onboarding,
    chunk_dense_with_onboarding
)
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

nest_asyncio.apply()

In [3]:
REPO_ROOT     = Path("/home/ubuntu/work/dahyeon")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data.json"

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

In [4]:
def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


In [25]:
# ============================================================
# Embedding Model별 Dense Retrieval 비교
# - isbn만 저장
# ============================================================

embedding_models = {
    "clova": "embedding",
    "kure": "embedding_kure",
    "bge_m3_ko": "embedding_bge_m3_ko",
}

all_candidates = []

for scenario in tqdm(scenarios, desc="시나리오 retrieval"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    for model_name, embedding_field in embedding_models.items():

        re_dense = full_dense_with_onboarding(
            sample_result,
            size=20,
            embedding_field=embedding_field
        )

        for rank, book in enumerate(re_dense, start=1):

            all_candidates.append({
                "query_id": query_id,
                "embedding_model": model_name,
                "embedding_field": embedding_field,
                "retrieval_type": "dense",
                "rank": rank,
                "isbn": book["isbn"]
            })

print(f"\n전체 후보 도서: {len(all_candidates):,}")

시나리오 retrieval: 100%|██████████| 21/21 [05:33<00:00, 15.88s/it]


전체 후보 도서: 1,260


In [16]:
# ============================================================
# Full vs Chunk Dense Retrieval 비교
# - embedding model: clova 고정
# - isbn만 저장
# ============================================================

retrieval_funcs = {
    "full": full_dense_with_onboarding,
    "chunk": chunk_dense_with_onboarding,
}

EMBEDDING_FIELD = "embedding"   # clova
CANDIDATE_SIZE = 100            # chunk는 ISBN 중복이 많아서 넉넉히 가져옴
TOP_K = 20

all_candidates = []

for scenario in tqdm(scenarios, desc="시나리오 retrieval"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    for retrieval_type, retrieval_func in retrieval_funcs.items():

        size = CANDIDATE_SIZE if retrieval_type == "chunk" else TOP_K

        re_dense = retrieval_func(
            sample_result,
            size=size,
            embedding_field=EMBEDDING_FIELD
        )

        # chunk는 같은 isbn이 여러 번 나올 수 있으므로 isbn 기준 dedup
        seen_isbns = set()
        rank = 1

        for book in re_dense:
            isbn = str(book["isbn"])

            if isbn in seen_isbns:
                continue

            seen_isbns.add(isbn)

            all_candidates.append({
                "query_id": query_id,
                "embedding_model": "clova",
                "embedding_field": EMBEDDING_FIELD,
                "retrieval_type": retrieval_type,   # full / chunk
                "rank": rank,
                "isbn": isbn
            })

            rank += 1

            if rank > TOP_K:
                break

print(f"\n전체 후보 도서: {len(all_candidates):,}")

dense_compare_df = pd.DataFrame(all_candidates)

dense_compare_df.head()

OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_dense_eval.jsonl")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:

    for item in all_candidates:
        f.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

print(f"저장 완료: {OUTPUT_PATH}")
print(f"총 저장 개수: {len(all_candidates):,}")

시나리오 retrieval: 100%|██████████| 21/21 [04:00<00:00, 11.47s/it]


전체 후보 도서: 840
저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/chunk_dense_eval.jsonl
총 저장 개수: 840


In [ ]:
import json
from pathlib import Path

OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/embedding_model_dense_eval.jsonl")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:

    for item in all_candidates:
        f.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

print(f"저장 완료: {OUTPUT_PATH}")
print(f"총 저장 개수: {len(all_candidates):,}")

저장 완료: /home/ubuntu/work/dahyeon/evaluation/datasetembedding_model_dense_eval.jsonl
총 저장 개수: 1,260


In [13]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# =====================================================
# 1. 파일 경로
# =====================================================

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final.json")
DENSE_RESULT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/embedding_model_dense_eval.jsonl")
REPO_ROOT = Path("/home/ubuntu/work/dahyeon")
sys.path.insert(0, str(REPO_ROOT / "evaluation" / "metrics"))

from metrics import hit_at_k, recall_at_k, mrr_at_k, ndcg_at_k



K_VALUES = [20]

# =====================================================
# 2. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

print("goldset 개수:", len(gold_df))
gold_df.head()

# =====================================================
# 3. dense retrieval 결과 로드
# =====================================================

dense_rows = []

with DENSE_RESULT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        dense_rows.append(json.loads(line))

dense_df = pd.DataFrame(dense_rows)

print("dense 결과 개수:", len(dense_df))
dense_df.head()

# =====================================================
# 4. goldset에서 query_id별 정답 ISBN 만들기
#    final_grade >= 2 를 relevant로 사용
# =====================================================

RELEVANCE_THRESHOLD = 2

relevant_by_query = {}

for qid, g in gold_df.groupby("query_id"):
    relevant_isbns = (
        g[g["final_grade"] >= RELEVANCE_THRESHOLD]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )
    relevant_by_query[qid] = set(relevant_isbns)

print("query 수:", len(relevant_by_query))
print("relevant 있는 query 수:", sum(1 for v in relevant_by_query.values() if v))

# =====================================================
# 5. query_id + embedding_model별 ranked ISBN 만들기
# =====================================================

eval_data = {}

for qid in relevant_by_query:
    eval_data[qid] = {
        "relevant_isbns": relevant_by_query[qid]
    }

for (qid, model), g in dense_df.groupby(["query_id", "embedding_model"]):
    ranked = (
        g.sort_values("rank")["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    if qid not in eval_data:
        eval_data[qid] = {
            "relevant_isbns": set()
        }

    eval_data[qid][model] = ranked
    
embedding_models = sorted(dense_df["embedding_model"].dropna().unique())
print("embedding models:", embedding_models)


# =====================================================
# 6. embedding model별 retrieval metric 계산
# =====================================================

def compute_embedding_model_metrics(
    eval_data,
    embedding_models,
    ks=K_VALUES,
):
    rows = []

    for qid, d in eval_data.items():
        rel = d["relevant_isbns"]

        if not rel:
            continue

        for model in embedding_models:
            ranked = d.get(model, [])

            if not ranked:
                continue

            for k in ks:
                rows.append({
                    "query_id": qid,
                    "embedding_model": model,
                    "k": k,
                    "hit": hit_at_k(rel, ranked, k),
                    "recall": recall_at_k(rel, ranked, k),
                    "mrr": mrr_at_k(rel, ranked, k),
                    "ndcg": ndcg_at_k(rel, ranked, k),
                })

    return pd.DataFrame(rows)


embedding_metric_df = compute_embedding_model_metrics(
    eval_data=eval_data,
    embedding_models=embedding_models,
    ks=K_VALUES,
)

print(f"총 {len(embedding_metric_df)}개 행 (시나리오 × embedding_model × K)")
embedding_metric_df.head(9)

goldset 개수: 1678
dense 결과 개수: 1260
query 수: 21
relevant 있는 query 수: 21
embedding models: ['bge_m3_ko', 'clova', 'kure']
총 63개 행 (시나리오 × embedding_model × K)


,query_id,embedding_model,k,hit,recall,mrr,ndcg
0,S01,bge_m3_ko,20,1,0.333333,0.250000,0.204356
1,S01,clova,20,1,0.166667,0.166667,0.107789
2,S01,kure,20,1,0.333333,0.250000,0.205974
3,S02,bge_m3_ko,20,0,0.000000,0.000000,0.000000
4,S02,clova,20,0,0.000000,0.000000,0.000000
5,S02,kure,20,0,0.000000,0.000000,0.000000
6,S03,bge_m3_ko,20,1,0.500000,0.500000,0.386853
7,S03,clova,20,1,1.000000,1.000000,0.877215
8,S03,kure,20,1,1.000000,0.500000,0.528722


In [14]:
# =====================================================
# 7. embedding_model × K 평균 성능
# =====================================================

summary = (
    embedding_metric_df
    .groupby(["embedding_model", "k"])[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
)

print("[embedding_model × K 평균 성능]")
print(summary.to_string())

[embedding_model × K 평균 성능]
                       hit  recall     mrr    ndcg
embedding_model k                                 
bge_m3_ko       20  0.8571  0.2932  0.4586  0.3158
clova           20  0.9048  0.3671  0.5282  0.3825
kure            20  0.9048  0.3474  0.3817  0.3353


# 청킹 전략 성능 비교

In [17]:
# =====================================================
# 1. 파일 경로
# =====================================================

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final.json")
DENSE_RESULT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/chunk_dense_eval.jsonl")

REPO_ROOT = Path("/home/ubuntu/work/dahyeon")
sys.path.insert(0, str(REPO_ROOT / "evaluation" / "metrics"))

from metrics import hit_at_k, recall_at_k, mrr_at_k, ndcg_at_k

K_VALUES = [20]
RELEVANCE_THRESHOLD = 2

# =====================================================
# 2. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

print("goldset 개수:", len(gold_df))
gold_df.head()

# =====================================================
# 3. full/chunk dense retrieval 결과 로드
# =====================================================

dense_rows = []

with DENSE_RESULT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        dense_rows.append(json.loads(line))

dense_df = pd.DataFrame(dense_rows)

print("dense 결과 개수:", len(dense_df))
dense_df.head()

# =====================================================
# 4. query_id별 정답 ISBN 만들기
#    final_grade >= 2 를 relevant로 사용
# =====================================================

relevant_by_query = {}

for qid, g in gold_df.groupby("query_id"):
    relevant_isbns = (
        g[g["final_grade"] >= RELEVANCE_THRESHOLD]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    relevant_by_query[qid] = set(relevant_isbns)

print("query 수:", len(relevant_by_query))
print("relevant 있는 query 수:", sum(1 for v in relevant_by_query.values() if v))

# =====================================================
# 5. eval_data 구성
#    retrieval_type: full / chunk
# =====================================================

eval_data = {}

for qid in relevant_by_query:
    eval_data[qid] = {
        "relevant_isbns": relevant_by_query[qid]
    }

for (qid, retrieval_type), g in dense_df.groupby(["query_id", "retrieval_type"]):
    ranked = (
        g.sort_values("rank")["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    if qid not in eval_data:
        eval_data[qid] = {
            "relevant_isbns": set()
        }

    eval_data[qid][retrieval_type] = ranked

retrieval_types = sorted(dense_df["retrieval_type"].dropna().unique())

print("retrieval types:", retrieval_types)

# =====================================================
# 6. full vs chunk retrieval metric 계산
# =====================================================

def compute_retrieval_metrics(
    eval_data,
    retrievers,
    ks=K_VALUES,
):
    rows = []

    for qid, d in eval_data.items():
        rel = d["relevant_isbns"]

        if not rel:
            continue

        for src in retrievers:
            ranked = d.get(src, [])

            if not ranked:
                continue

            for k in ks:
                rows.append({
                    "query_id": qid,
                    "retriever": src,
                    "k": k,
                    "hit": hit_at_k(rel, ranked, k),
                    "recall": recall_at_k(rel, ranked, k),
                    "mrr": mrr_at_k(rel, ranked, k),
                    "ndcg": ndcg_at_k(rel, ranked, k),
                })

    return pd.DataFrame(rows)


retrieval_df = compute_retrieval_metrics(
    eval_data=eval_data,
    retrievers=retrieval_types,
    ks=K_VALUES,
)

print(f"총 {len(retrieval_df)}개 행 (시나리오 × retrieval_type × K)")
retrieval_df.head(9)

goldset 개수: 1678
dense 결과 개수: 840
query 수: 21
relevant 있는 query 수: 21
retrieval types: ['chunk', 'full']
총 42개 행 (시나리오 × retrieval_type × K)


,query_id,retriever,k,hit,recall,mrr,ndcg
0,S01,chunk,20,1,0.333333,0.125000,0.169492
1,S01,full,20,1,0.166667,0.166667,0.107789
2,S02,chunk,20,1,0.333333,0.142857,0.156426
3,S02,full,20,1,0.333333,0.083333,0.126817
4,S03,chunk,20,0,0.000000,0.000000,0.000000
5,S03,full,20,1,0.500000,0.142857,0.204382
6,S04,chunk,20,1,0.625000,0.500000,0.493419
7,S04,full,20,1,0.625000,1.000000,0.558907
8,S05,chunk,20,0,0.000000,0.000000,0.000000


In [18]:
# =====================================================
# 7. full vs chunk 평균 성능
# =====================================================

summary = (
    retrieval_df
    .groupby(["retriever", "k"])[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
)

print("[full vs chunk 평균 성능]")
print(summary.to_string())

[full vs chunk 평균 성능]
                 hit  recall     mrr    ndcg
retriever k                                 
chunk     20  0.8095  0.2484  0.3422  0.2502
full      20  0.9524  0.3591  0.4914  0.3576
